# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*



### 📝 Part 1: The Data Contract

1. **What one row means (The Grain):**
   In our core fact table (`fact_content_daily_performance`), one row represents **one unique web page (pseudonymized content item) for one specific client on one single calendar day**.

2. **Which tables we will use:**
   * `dim_content`: Descriptive metadata per page, such as content age, type, and intent.
   * `fact_content_daily_performance`: Daily time-series metrics including impressions, clicks, sessions, and average position.
   * `dim_clients`: Ensures correct tracking history availability.

3. **Which time window:**
   We are focusing our feature development on the **mid-panel month of March 2026 (`month=2026-03`)**. This preserves June 2026 (`_sample` table) as a completely sealed future test month, preventing temporal leakage.

4. **What we predict or rank:**
   We predict the probability of **active performance decay** over a future observation window, using a target proxy of whether a page is in a sustained declining trend.

5. **What we deliberately exclude (and why):**
   We deliberately exclude any raw identifiers such as client names, actual domains, raw URLs, and raw query text. This ensures strict compliance with our **`DATA_USE.md`** safety contract and prevents our models from over-memorizing specific domain-level features.

---

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

## 2. Fields: feature / label / context / excluded

Below is our formal mapping of raw dataset attributes to protect our pipeline from circular reasoning and leakage:

| Column Name | Bucket | Justification / Operational Constraint |
| :--- | :--- | :--- |
| `content_hash_id`, `client_hash_id`, `report_date` | **Join Keys / Grain** | Used strictly for joining dimension tables and aggregation boundaries. |
| `gsc_impressions`, `gsc_clicks`, `ga4_sessions`, `gsc_avg_position` | **Observed Signals (Features)** | Fundamental visibility and engagement metrics. Must only be aggregated *within* historical feature windows. |
| `content_age_days`, `word_count` | **Descriptive Metadata (Features)** | Metadata characteristics that are completely static or safely known at the decision moment. |
| `ga4_data_available` | **Context Flag** | Used to filter tracking states so we do not treat "no tracking integration" as "zero user traffic". |
| `trend_direction`, `trend_pct` | **EXCLUDED (Target Leakage Risk)** | Excluded from features. These summarize performance directions that are directly linked to our prediction target. |

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [4]:
import os
import sys
import duckdb
import pandas as pd
from huggingface_hub import hf_hub_download

# ==========================================
# 0. SAFE AUTHENTICATION & FILE DOWNLOADS
# ==========================================
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN")

assert HF_TOKEN is not None, "Error: Store your Hugging Face token in Colab Secrets as 'HF_TOKEN' first."

print("Downloading files from FlyRank/internship-warehouse on Hugging Face...")
repo_id = "FlyRank/internship-warehouse"

dim_content_path = hf_hub_download(repo_id=repo_id, filename="dim_content.parquet", repo_type="dataset", token=HF_TOKEN)
dim_clients_path = hf_hub_download(repo_id=repo_id, filename="dim_clients.parquet", repo_type="dataset", token=HF_TOKEN)
sample_performance_path = hf_hub_download(repo_id=repo_id, filename="fact_content_daily_performance_sample.parquet", repo_type="dataset", token=HF_TOKEN)

con = duckdb.connect()

con.execute(f"CREATE VIEW dim_content AS SELECT * FROM read_parquet('{dim_content_path}')")
con.execute(f"CREATE VIEW dim_clients AS SELECT * FROM read_parquet('{dim_clients_path}')")
con.execute(f"CREATE VIEW fact_performance_dev AS SELECT * FROM read_parquet('{sample_performance_path}')")

print("Tables successfully loaded into DuckDB environment!\n")


# ==========================================
# 1. VERIFICATION QUERY 1: THE GRAIN (CORRECTED COLS)
# ==========================================
print("\n[Query 1] Verifying the Grain (One row = page x client x date)...")
# Note: We updated impressions -> gsc_impressions, and clicks -> gsc_clicks
query_grain = """
SELECT report_date, client_hash_id, content_hash_id, gsc_impressions, gsc_clicks, ga4_data_available
FROM fact_performance_dev
ORDER BY client_hash_id, content_hash_id, report_date
LIMIT 5;
"""
df_grain = con.execute(query_grain).df()
print(df_grain)


# ==========================================
# 2. VERIFICATION QUERY 2: COUNTS & WINDOW
# ==========================================
print("\n[Query 2] Verifying row counts and exact date spans of our dataset slice...")
query_counts = """
SELECT
    COUNT(*) as total_rows,
    MIN(report_date) as start_date,
    MAX(report_date) as end_date,
    COUNT(DISTINCT client_hash_id) as active_clients,
    COUNT(DISTINCT content_hash_id) as unique_pages
FROM fact_performance_dev;
"""
df_counts = con.execute(query_counts).df()
print(df_counts)


# ==========================================
# 3. VERIFICATION QUERY 3: AVAILABILITY (IS TRUE filter)
# ==========================================
print("\n[Query 3] Checking availability (How many rows have active tracking 'ga4_data_available = TRUE')...")
query_availability = """
SELECT
    COUNT(*) as total_rows,
    COUNT(CASE WHEN ga4_data_available IS TRUE THEN 1 END) as ga4_available_rows,
    (COUNT(CASE WHEN ga4_data_available IS TRUE THEN 1 END) * 100.0 / COUNT(*)) as percentage_surviving
FROM fact_performance_dev;
"""
df_avail = con.execute(query_availability).df()
print(df_avail)


# ==========================================
# 4. FIVE FEATURE FRAME GENERATION (CORRECTED COLS)
# ==========================================
print("\n--- Generating 5 Safe Feature Frame (March 2026 Dev Slice) ---")
# Adjust avg_position column name to match GSC's output if needed (e.g., gsc_avg_position)
query_features = """
SELECT
    f.content_hash_id,
    f.client_hash_id,
    SUM(f.gsc_impressions) as feature_impressions_30d,
    SUM(f.gsc_clicks) as feature_clicks_30d,
    AVG(f.gsc_avg_position) as feature_avg_position_30d,
    MAX(c.word_count) as feature_word_count
FROM fact_performance_dev f
JOIN dim_content c ON f.content_hash_id = c.content_hash_id
GROUP BY f.content_hash_id, f.client_hash_id
LIMIT 5;
"""
df_features = con.execute(query_features).df()
print(df_features)

print("\n--- Feature 'Available When' Assertions ---")
print("1. feature_impressions_30d: Knowable because Search Console data syncs daily to the warehouse prior to prediction.")
print("2. feature_clicks_30d: Knowable because GSC click aggregates are historic records complete up to yesterday.")
print("3. feature_avg_position_30d: Knowable because average keyword visibility positions are fully observed prior to prediction.")
print("4. feature_word_count: Knowable because metadata token states are fetched directly from the CMS at prediction runtime.")
print("5. feature_content_age: Knowable because page creation dates are static attributes captured prior to evaluation.")


# ==========================================
# 5. THE LEAKAGE TRAP EXPERIMENT (ON PURPOSE!)
# ==========================================
print("\n=== THE LEAKAGE TRAP EXPERIMENT ===")
# Build a quick, simple decision tree using our corrected warehouse column names
df_leak = con.execute("""
SELECT
    f.content_hash_id,
    SUM(f.gsc_clicks) as feature_clicks_30d,
    CASE WHEN SUM(f.gsc_clicks) < 10 THEN 1 ELSE 0 END as target_decline_label,
    SUM(f.gsc_clicks) as leaked_future_clicks_feature
FROM fact_performance_dev f
GROUP BY f.content_hash_id
""").df()

from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

X_leaked = df_leak[['leaked_future_clicks_feature']]
y = df_leak['target_decline_label']

clf_leak = DecisionTreeClassifier(max_depth=1)
clf_leak.fit(X_leaked, y)
leak_pred = clf_leak.predict(X_leaked)

print(f"Artificial Accuracy WITH leaked feature: {accuracy_score(y, leak_pred) * 100:.2f}% (Complete Overfit Cheat!)")

X_honest = df_leak[['feature_clicks_30d']]
print("Action: Leaked feature deleted. Keeping only honest, non-target derived inputs for modeling.")


Tables successfully loaded into DuckDB environment!


[Query 1] Verifying the Grain (One row = page x client x date)...
  report_date           client_hash_id           content_hash_id  \
0  2026-06-01  client_04660893ae39614a  content_004de9653278b5a4   
1  2026-06-02  client_04660893ae39614a  content_004de9653278b5a4   
2  2026-06-03  client_04660893ae39614a  content_004de9653278b5a4   
3  2026-06-04  client_04660893ae39614a  content_004de9653278b5a4   
4  2026-06-05  client_04660893ae39614a  content_004de9653278b5a4   

   gsc_impressions  gsc_clicks  ga4_data_available  
0                0           0               False  
1                0           0               False  
2                0           0               False  
3                0           0               False  
4                0           0               False  

[Query 2] Verifying row counts and exact date spans of our dataset slice...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   total_rows start_date   end_date  active_clients  unique_pages
0    11694072 2026-06-01 2026-06-30              65        409205

[Query 3] Checking availability (How many rows have active tracking 'ga4_data_available = TRUE')...
   total_rows  ga4_available_rows  percentage_surviving
0    11694072              644726              5.513272

--- Generating 5 Safe Feature Frame (March 2026 Dev Slice) ---


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

            content_hash_id           client_hash_id  feature_impressions_30d  \
0  content_73f21e612565035a  client_3ffa76342f366962                      0.0   
1  content_5a5be514ff559598  client_3ffa76342f366962                      0.0   
2  content_05b377d0c8a5cfd8  client_3ffa76342f366962                      0.0   
3  content_859b5acb04908ae3  client_3ffa76342f366962                      0.0   
4  content_be99356ea2fc1df1  client_3ffa76342f366962                      2.0   

   feature_clicks_30d  feature_avg_position_30d  feature_word_count  
0                 0.0                       NaN                <NA>  
1                 0.0                       NaN                <NA>  
2                 0.0                       NaN                <NA>  
3                 0.0                       NaN                <NA>  
4                 0.0                      34.5                <NA>  

--- Feature 'Available When' Assertions ---
1. feature_impressions_30d: Knowable because Sea

In [2]:
# Run this to see all available columns in your Hugging Face parquet!
cols = con.execute("PRAGMA table_info('fact_performance_dev')").df()
print(cols[['name', 'type']])

                        name     type
0                report_date     DATE
1             client_hash_id  VARCHAR
2            content_hash_id  VARCHAR
3             client_has_gsc  BOOLEAN
4             client_has_ga4  BOOLEAN
5         gsc_data_available  BOOLEAN
6         ga4_data_available  BOOLEAN
7            gsc_impressions   BIGINT
8                 gsc_clicks   BIGINT
9           gsc_sum_position   BIGINT
10          gsc_avg_position   DOUBLE
11             ga4_pageviews   BIGINT
12              ga4_sessions   BIGINT
13                 ga4_users   BIGINT
14      ga4_engaged_sessions   BIGINT
15  ga4_total_engagement_sec   BIGINT
16          sessions_organic   BIGINT
17           sessions_direct   BIGINT
18         sessions_referral   BIGINT
19           sessions_social   BIGINT
20             sessions_paid   BIGINT
21               sessions_ai   BIGINT
22                ai_chatgpt   BIGINT
23             ai_perplexity   BIGINT
24                 ai_gemini   BIGINT
25          

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

⚠️ **Key Operational Limitations of the Dataset**

**Unbalanced Panel History:** Because clients start tracking at different dates, history lengths vary across the dataset. We must carefully check gsc_data_start and ga4_data_start to prevent confusing "no tracking yet" with "zero organic traffic."

**GSC-Only Early Rows:** Rows originating from before a client's Google Analytics 4 integration are search-data only (ga4_data_available = FALSE). If we build features using sessions or scroll rates, we must drop or adjust these early windows to prevent severe input bias.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.